In [0]:
gold_bt = spark.table("workspace.default.silver_bank_transactions")

In [0]:
from pyspark.sql.functions import countDistinct, count_if, col, stddev, avg, variance, corr, percentile_approx, max, first, last

aggregation

In [0]:
gold_bt.agg(countDistinct("CustLocation")).show()

In [0]:
gold_bt.printSchema()

In [0]:
gold_bt.agg(countDistinct("CustomerID")).show()

In [0]:
gold_bt.agg(
    countDistinct("CustomerID").alias("DistinctCustomers"),
    countDistinct("CustLocation").alias("DistinctLocations")
).show()

In [0]:
gold_bt.agg(count_if(col("TransactionAmountINR") > 50000)).show()

In [0]:
gold_bt.agg(count_if(col("TransactionAmountINR") < 1000).alias("LowBalanceTxns")).show()

Stats

measures how spread out values are around the average. A low stddev means most values cluster close to the mean; a high one means they're scattered widely.

In [0]:
gold_bt.agg(
    avg(col("TransactionAmountINR")).alias("AvgTxnAmt"),
    stddev(col("TransactionAmountINR"))
).show()

In [0]:
gold_bt.agg(variance(col("TransactionAmountINR"))).show()

corr() only works on numeric columns, because correlation is fundamentally a mathematical relationship between quantities

In [0]:
gold_bt.agg(corr(col("CustAccountBalance"), col("TransactionAmountINR"))).show()

position-based functions — first(), last(), percentile_approx().

In [0]:
gold_bt.agg(percentile_approx(col("TransactionAmountINR"), 0.5)).show()

In [0]:
gold_bt.agg(percentile_approx(col("TransactionAmountINR"), [0.25, 0.5, 0.75])).show()

first() and last().

What they do: within a group (or across a whole DataFrame), grab the first or last value encountered — not the biggest, not the smallest, just whichever row happens to come first or last in whatever order Spark processes the data.

The important warning, worth internalizing before you ever use these for real: unless you explicitly sort the data first, "first" and "last" have no reliable meaning. Spark processes data across a distributed cluster, in whatever order is most efficient — there's no guaranteed "natural order" to a table the way there might be in a spreadsheet. So first() without an explicit orderBy() beforehand can give you a different row every time you run the exact same query.

In [0]:
gold_bt.groupBy("CustLocation").agg(max(col("TransactionDate")).alias("MostRecentTxn")).show(10)

explicitly sort the data by date (most recent first) before grouping, else first() will give different answers each time

In [0]:
gold_bt.orderBy(col("TransactionDate")).groupBy("CustLocation").agg(first(col("TransactionDate"))).show()

The actual lesson here, worth remembering as a rule rather than a one-off fix: max()/min() are the correct, trustworthy tools whenever you genuinely want the largest/smallest value — they're mathematically defined and don't depend on row order at all. first()/last() should really only be used when you deliberately want "whichever row happens to represent this group"